Mapping Fertility Services in the United Kingdom: A Web Scraping Analysis of HFEA Clinic Profiles

In [3]:
from selenium import webdriver
from bs4 import BeautifulSoup

# Open Chrome
driver = webdriver.Chrome()

# Navigate to the alphabetical clinic directory
starting_url = "https://www.hfea.gov.uk/choose-a-fertility-clinic/search/all-clinics/?alpha=a"

driver.get(starting_url)

# Find the clinic headings
clinic_headings = driver.find_elements("xpath", "//h2")

print("Number of h2 elements:", len(clinic_headings))

for heading in clinic_headings[:20]:
    print(heading.text)

Number of h2 elements: 22
Abbey Fertility Clinic
Aberdeen Fertility Centre
Acorn Fertility
Adrian Lower
Agora Clinic Brighton
Agora Clinic Eastbourne
Agora Clinic Worthing
Alma Fertility
Andrology Solutions
Andrology Unit, Hammersmith Hospital
Aria Fertility
Assisted Reproduction and Gynaecology Centre
Assisted Reproduction Unit (ARU), University Hospital of Hartlepool
Assisted Reproduction Unit, Jersey General Hospital
Aurora Reproductive Healthcare
Avenues
Avenues Cambridge
Ayrshire Fertility Unit, University Hospital Crosshouse
Get information
About the HFEA


In [4]:
# Find Aberdeen Fertility Centre
clinic = driver.find_element(
    "xpath",
    "//h2[contains(., 'Aberdeen Fertility Centre')]"
)

# Look at the HTML around this clinic heading
print(clinic.get_attribute("outerHTML"))

# Check the surrounding container
print("\nPARENT ELEMENT:\n")
print(clinic.find_element("xpath", "..").get_attribute("outerHTML")[:3000])

<h2 class="h3 mb-40"><a href="/choose-a-fertility-clinic/search/results/19-aberdeen-fertility-centre/">Aberdeen Fertility Centre</a></h2>

PARENT ELEMENT:

<div class="col-md-12">
                                    <h2 class="h3 mb-40"><a href="/choose-a-fertility-clinic/search/results/19-aberdeen-fertility-centre/">Aberdeen Fertility Centre</a></h2>

                            </div>


In [5]:
# Go up two levels from the clinic heading
clinic_container = clinic.find_element("xpath", "../..")

print(clinic_container.get_attribute("outerHTML")[:5000])

<div class="row">
                            <div class="col-md-12">
                                    <h2 class="h3 mb-40"><a href="/choose-a-fertility-clinic/search/results/19-aberdeen-fertility-centre/">Aberdeen Fertility Centre</a></h2>

                            </div>
                            <div class="col-md-6 list">
                                    <ul class="list-unstyled">
                                            <li>Department of Obstetrics &amp; Gynaecology</li>
                                            <li> Aberdeen Maternity Hospital</li>
                                            <li> Foresterhill</li>
                                            <li> Aberdeen</li>
                                            <li> AB25 2ZL</li>
                                            <li> Scotland</li>
                                    </ul>
                            </div>
                            <div class="col-md-6 list">
                                <u

In [10]:
from urllib.parse import urljoin

clinic_containers = driver.find_elements(
    "xpath",
    "//div[contains(@class, 'row')][./div[contains(@class, 'col-md-12')]/h2[contains(@class, 'h3')]]"
)

print("Number of clinic containers:", len(clinic_containers))

Number of clinic containers: 18


In [11]:
for container in clinic_containers:
    print(container.find_element("xpath", "./div/h2").text)

Abbey Fertility Clinic
Aberdeen Fertility Centre
Acorn Fertility
Adrian Lower
Agora Clinic Brighton
Agora Clinic Eastbourne
Agora Clinic Worthing
Alma Fertility
Andrology Solutions
Andrology Unit, Hammersmith Hospital
Aria Fertility
Assisted Reproduction and Gynaecology Centre
Assisted Reproduction Unit (ARU), University Hospital of Hartlepool
Assisted Reproduction Unit, Jersey General Hospital
Aurora Reproductive Healthcare
Avenues
Avenues Cambridge
Ayrshire Fertility Unit, University Hospital Crosshouse


In [ ]:
from urllib.parse import urljoin
import pandas as pd

clinics = []

for container in clinic_containers:

    # Clinic name
    name = container.find_element("xpath", "./div/h2").text

    # HFEA profile link
    links = container.find_elements("xpath", "./div/h2/a")

    if links:
        profile_url = links[0].get_attribute("href")
    else:
        profile_url = None

    # Address
    address_elements = container.find_elements(
        "xpath",
        "./div[contains(@class, 'col-md-6')][1]//li"
    )

    address_parts = [element.text.strip() for element in address_elements]

    # Save the clinic information
    clinics.append({
        "clinic_name": name,
        "address": ", ".join(address_parts),
        "postcode": address_parts[-2] if len(address_parts) >= 2 else None,
        "country": address_parts[-1] if address_parts else None,
        "profile_url": profile_url
    })

clinics_df = pd.DataFrame(clinics)

display(clinics_df)



,clinic_name,address,postcode,country,profile_url
0,Abbey Fertility Clinic,"6 Telford Court, Chestergates Business Park, C...",CH1 6LT,,None
1,Aberdeen Fertility Centre,"Department of Obstetrics & Gynaecology, Aberde...",AB25 2ZL,Scotland,https://www.hfea.gov.uk/choose-a-fertility-cli...
2,Acorn Fertility,"112a, Cavendish Road, Colliers Wood, Merton, S...",SW19 2EZ,,https://www.hfea.gov.uk/choose-a-fertility-cli...
3,Adrian Lower,"9, Upper Wimpole Street, London, W1G 6LJ,",W1G 6LJ,,None
4,Agora Clinic Brighton,"138, Dyke Road, Brighton, BN1 5PA,",BN1 5PA,,https://www.hfea.gov.uk/choose-a-fertility-cli...
5,Agora Clinic Eastbourne,"J7, J8 Franklin House, Chaucer Business Park, ...",BN26 6JF,,https://www.hfea.gov.uk/choose-a-fertility-cli...
6,Agora Clinic Worthing,"Phoenix Medical Centre, Lime Tree Surgery Gori...",BN124FD,,None
7,Alma Fertility,"LIPS Healthcare, , Battersea Power Station, Le...",SW11 8DD,,None
8,Andrology Solutions,"75, Wimpole Street, London, W1G 9RT,",W1G 9RT,,https://www.hfea.gov.uk/choose-a-fertility-cli...
9,"Andrology Unit, Hammersmith Hospital","South Corridor, Area C/FR30, Hammersmith Hospi...",W12 0HS,United Kingdom,https://www.hfea.gov.uk/choose-a-fertility-cli...


In [13]:
# Keep the original data but remove the inconsistent country variable
clinics_df = clinics_df.drop(columns=["country"])

# Indicate whether a clinic has an individual HFEA profile
clinics_df["has_profile"] = clinics_df["profile_url"].notna()

display(clinics_df.head())

print("\nClinics with profiles:", clinics_df["has_profile"].sum())
print("Clinics without profiles:", (~clinics_df["has_profile"]).sum())

,clinic_name,address,postcode,profile_url,has_profile
0,Abbey Fertility Clinic,"6 Telford Court, Chestergates Business Park, C...",CH1 6LT,None,False
1,Aberdeen Fertility Centre,"Department of Obstetrics & Gynaecology, Aberde...",AB25 2ZL,https://www.hfea.gov.uk/choose-a-fertility-cli...,True
2,Acorn Fertility,"112a, Cavendish Road, Colliers Wood, Merton, S...",SW19 2EZ,https://www.hfea.gov.uk/choose-a-fertility-cli...,True
3,Adrian Lower,"9, Upper Wimpole Street, London, W1G 6LJ,",W1G 6LJ,None,False
4,Agora Clinic Brighton,"138, Dyke Road, Brighton, BN1 5PA,",BN1 5PA,https://www.hfea.gov.uk/choose-a-fertility-cli...,True



Clinics with profiles: 11
Clinics without profiles: 7


In [14]:
import requests
import pandas as pd

geo_data = []

for postcode in clinics_df["postcode"]:

    url = "https://api.postcodes.io/postcodes/" + postcode.replace(" ", "")
    response = requests.get(url, timeout=15)

    if response.status_code == 200:
        result = response.json()["result"]
    else:
        result = None

    if result:
        geo_data.append({
            "postcode": postcode,
            "local_authority": result["admin_district"],
            "local_authority_code": result["codes"]["admin_district"],
            "country": result["country"],
            "latitude": result["latitude"],
            "longitude": result["longitude"]
        })
    else:
        geo_data.append({
            "postcode": postcode,
            "local_authority": None,
            "local_authority_code": None,
            "country": None,
            "latitude": None,
            "longitude": None
        })

geo_df = pd.DataFrame(geo_data)

display(geo_df)

,postcode,local_authority,local_authority_code,country,latitude,longitude
0,CH1 6LT,Cheshire West and Chester,E06000050,England,53.246752,-2.932817
1,AB25 2ZL,Aberdeen City,S12000033,Scotland,57.155245,-2.132080
2,SW19 2EZ,Merton,E09000024,England,51.418616,-0.172276
3,W1G 6LJ,Westminster,E09000033,England,51.521066,-0.149078
4,BN1 5PA,Brighton and Hove,E06000043,England,50.834805,-0.150779
5,BN26 6JF,Wealden,E07000065,England,50.819678,0.268208
6,BN124FD,Worthing,E07000229,England,50.806603,-0.404420
7,SW11 8DD,Wandsworth,E09000032,England,51.481512,-0.145438
8,W1G 9RT,Westminster,E09000033,England,51.517464,-0.148200
9,W12 0HS,Hammersmith and Fulham,E09000013,England,51.517422,-0.234698


In [15]:
clinics_geo_df = clinics_df.merge(
    geo_df,
    on="postcode",
    how="left"
)

display(clinics_geo_df)

,clinic_name,address,postcode,profile_url,has_profile,local_authority,local_authority_code,country,latitude,longitude
0,Abbey Fertility Clinic,"6 Telford Court, Chestergates Business Park, C...",CH1 6LT,None,False,Cheshire West and Chester,E06000050,England,53.246752,-2.932817
1,Aberdeen Fertility Centre,"Department of Obstetrics & Gynaecology, Aberde...",AB25 2ZL,https://www.hfea.gov.uk/choose-a-fertility-cli...,True,Aberdeen City,S12000033,Scotland,57.155245,-2.132080
2,Acorn Fertility,"112a, Cavendish Road, Colliers Wood, Merton, S...",SW19 2EZ,https://www.hfea.gov.uk/choose-a-fertility-cli...,True,Merton,E09000024,England,51.418616,-0.172276
3,Adrian Lower,"9, Upper Wimpole Street, London, W1G 6LJ,",W1G 6LJ,None,False,Westminster,E09000033,England,51.521066,-0.149078
4,Agora Clinic Brighton,"138, Dyke Road, Brighton, BN1 5PA,",BN1 5PA,https://www.hfea.gov.uk/choose-a-fertility-cli...,True,Brighton and Hove,E06000043,England,50.834805,-0.150779
5,Agora Clinic Eastbourne,"J7, J8 Franklin House, Chaucer Business Park, ...",BN26 6JF,https://www.hfea.gov.uk/choose-a-fertility-cli...,True,Wealden,E07000065,England,50.819678,0.268208
6,Agora Clinic Worthing,"Phoenix Medical Centre, Lime Tree Surgery Gori...",BN124FD,None,False,Worthing,E07000229,England,50.806603,-0.404420
7,Alma Fertility,"LIPS Healthcare, , Battersea Power Station, Le...",SW11 8DD,None,False,Wandsworth,E09000032,England,51.481512,-0.145438
8,Andrology Solutions,"75, Wimpole Street, London, W1G 9RT,",W1G 9RT,https://www.hfea.gov.uk/choose-a-fertility-cli...,True,Westminster,E09000033,England,51.517464,-0.148200
9,"Andrology Unit, Hammersmith Hospital","South Corridor, Area C/FR30, Hammersmith Hospi...",W12 0HS,https://www.hfea.gov.uk/choose-a-fertility-cli...,True,Hammersmith and Fulham,E09000013,England,51.517422,-0.234698


In [16]:
print("Number of clinics:", len(clinics_geo_df))

print("\nMissing geographic information:")
print(
    clinics_geo_df[
        ["local_authority", "country", "latitude", "longitude"]
    ].isna().sum()
)

Number of clinics: 18

Missing geographic information:
local_authority    0
country            0
latitude           1
longitude          1
dtype: int64


In [17]:
import string
import time
import pandas as pd

from selenium import webdriver

driver = webdriver.Chrome()

# Start with A, B, and C
letters = string.ascii_lowercase[:3]

all_clinics = []

for letter in letters:

    url = f"https://www.hfea.gov.uk/choose-a-fertility-clinic/search/all-clinics/?alpha={letter}"

    driver.get(url)
    time.sleep(5)

    # Find individual clinic containers
    clinic_containers = driver.find_elements(
        "xpath",
        "//div[contains(@class, 'row')][./div[contains(@class, 'col-md-12')]/h2[contains(@class, 'h3')]]"
    )

    print(f"Letter {letter.upper()}: {len(clinic_containers)} clinics")

    for container in clinic_containers:

        # Clinic name
        name = container.find_element("xpath", "./div/h2").text

        # HFEA profile link
        profile_links = container.find_elements("xpath", "./div/h2/a")

        if profile_links:
            profile_url = profile_links[0].get_attribute("href")
        else:
            profile_url = None

        # Address
        address_elements = container.find_elements(
            "xpath",
            "./div[contains(@class, 'col-md-6')][1]//li"
        )

        address_parts = [element.text.strip() for element in address_elements]

        address = ", ".join(address_parts)

        postcode = address_parts[-2] if len(address_parts) >= 2 and address_parts[-1] in ["England", "Scotland", "Wales", "Northern Ireland", "United Kingdom"] else (
            address_parts[-1] if address_parts else None
        )

        # Clinic website
        website_links = container.find_elements(
            "xpath",
            ".//a[contains(@title, 'Website link')]"
        )

        if website_links:
            website = website_links[0].get_attribute("href")
        else:
            website = None

        all_clinics.append({
            "clinic_name": name,
            "address": address,
            "postcode": postcode,
            "profile_url": profile_url,
            "website": website
        })

# Convert to DataFrame
all_clinics_df = pd.DataFrame(all_clinics)

display(all_clinics_df.head(10))

print("\nTotal clinics:", len(all_clinics_df))
print("Duplicate clinic names:", all_clinics_df["clinic_name"].duplicated().sum())

Letter A: 18 clinics
Letter B: 12 clinics
Letter C: 47 clinics


,clinic_name,address,postcode,profile_url,website
0,Abbey Fertility Clinic,"6 Telford Court, Chestergates Business Park, C...",,None,None
1,Aberdeen Fertility Centre,"Department of Obstetrics & Gynaecology, Aberde...",AB25 2ZL,https://www.hfea.gov.uk/choose-a-fertility-cli...,http://www.aberdeenfertility.org.uk/
2,Acorn Fertility,"112a, Cavendish Road, Colliers Wood, Merton, S...",,https://www.hfea.gov.uk/choose-a-fertility-cli...,None
3,Adrian Lower,"9, Upper Wimpole Street, London, W1G 6LJ,",,None,None
4,Agora Clinic Brighton,"138, Dyke Road, Brighton, BN1 5PA,",,https://www.hfea.gov.uk/choose-a-fertility-cli...,http://www.agoraclinic.co.uk/
5,Agora Clinic Eastbourne,"J7, J8 Franklin House, Chaucer Business Park, ...",,https://www.hfea.gov.uk/choose-a-fertility-cli...,https://agoraclinic.co.uk/
6,Agora Clinic Worthing,"Phoenix Medical Centre, Lime Tree Surgery Gori...",,None,None
7,Alma Fertility,"LIPS Healthcare, , Battersea Power Station, Le...",,None,None
8,Andrology Solutions,"75, Wimpole Street, London, W1G 9RT,",,https://www.hfea.gov.uk/choose-a-fertility-cli...,http://www.andrologysolutions.co.uk/
9,"Andrology Unit, Hammersmith Hospital","South Corridor, Area C/FR30, Hammersmith Hospi...",W12 0HS,https://www.hfea.gov.uk/choose-a-fertility-cli...,None



Total clinics: 77
Duplicate clinic names: 0


In [18]:
print("\nMissing values:")
print(all_clinics_df.isna().sum())

print("\nPostcodes:")
display(all_clinics_df[["clinic_name", "postcode"]].head(20))


Missing values:
clinic_name     0
address         0
postcode        0
profile_url    32
website        40
dtype: int64

Postcodes:


,clinic_name,postcode
0,Abbey Fertility Clinic,
1,Aberdeen Fertility Centre,AB25 2ZL
2,Acorn Fertility,
3,Adrian Lower,
4,Agora Clinic Brighton,
5,Agora Clinic Eastbourne,
6,Agora Clinic Worthing,
7,Alma Fertility,
8,Andrology Solutions,
9,"Andrology Unit, Hammersmith Hospital",W12 0HS


In [19]:
driver.get(
    "https://www.hfea.gov.uk/choose-a-fertility-clinic/search/all-clinics/?alpha=a"
)

clinic_containers = driver.find_elements(
    "xpath",
    "//div[contains(@class, 'row')][./div[contains(@class, 'col-md-12')]/h2[contains(@class, 'h3')]]"
)

abbey = clinic_containers[0]

address_elements = abbey.find_elements(
    "xpath",
    "./div[contains(@class, 'col-md-6')][1]//li"
)

print([element.text for element in address_elements])

['6 Telford Court', 'Chestergates Business Park', 'Chester', 'CH1 6LT', '']


In [21]:
import re

def extract_postcode(address_parts):

    address_text = " ".join(address_parts)

    pattern = r"\b[A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2}\b"

    match = re.search(pattern, address_text, re.IGNORECASE)

    if match:
        return match.group().upper()
    else:
        return None

In [22]:
all_clinics_df["postcode"] = all_clinics_df["address"].apply(
    lambda address: extract_postcode([address])
)

display(all_clinics_df[["clinic_name", "postcode"]].head(20))

print("\nMissing postcodes:", all_clinics_df["postcode"].isna().sum())

,clinic_name,postcode
0,Abbey Fertility Clinic,CH1 6LT
1,Aberdeen Fertility Centre,AB25 2ZL
2,Acorn Fertility,SW19 2EZ
3,Adrian Lower,W1G 6LJ
4,Agora Clinic Brighton,BN1 5PA
5,Agora Clinic Eastbourne,BN26 6JF
6,Agora Clinic Worthing,BN124FD
7,Alma Fertility,SW11 8DD
8,Andrology Solutions,W1G 9RT
9,"Andrology Unit, Hammersmith Hospital",W12 0HS



Missing postcodes: 0


In [23]:
import string
import time
import pandas as pd

# All letters from A to Z
letters = string.ascii_lowercase

all_clinics = []

for letter in letters:

    url = f"https://www.hfea.gov.uk/choose-a-fertility-clinic/search/all-clinics/?alpha={letter}"

    driver.get(url)
    time.sleep(5)

    clinic_containers = driver.find_elements(
        "xpath",
        "//div[contains(@class, 'row')][./div[contains(@class, 'col-md-12')]/h2[contains(@class, 'h3')]]"
    )

    print(f"Letter {letter.upper()}: {len(clinic_containers)} clinics")

    for container in clinic_containers:

        # Clinic name
        name = container.find_element("xpath", "./div/h2").text

        # HFEA profile link
        profile_links = container.find_elements("xpath", "./div/h2/a")

        if profile_links:
            profile_url = profile_links[0].get_attribute("href")
        else:
            profile_url = None

        # Address
        address_elements = container.find_elements(
            "xpath",
            "./div[contains(@class, 'col-md-6')][1]//li"
        )

        address_parts = [
            element.text.strip()
            for element in address_elements
            if element.text.strip()
        ]

        address = ", ".join(address_parts)
        postcode = extract_postcode(address_parts)

        # Clinic website
        website_links = container.find_elements(
            "xpath",
            ".//a[contains(@title, 'Website link')]"
        )

        if website_links:
            website = website_links[0].get_attribute("href")
        else:
            website = None

        all_clinics.append({
            "clinic_name": name,
            "address": address,
            "postcode": postcode,
            "profile_url": profile_url,
            "website": website
        })

all_clinics_df = pd.DataFrame(all_clinics)

print("\nTotal clinic listings:", len(all_clinics_df))
display(all_clinics_df.head())

Letter A: 18 clinics
Letter B: 12 clinics
Letter C: 47 clinics
Letter D: 5 clinics
Letter E: 3 clinics
Letter F: 13 clinics
Letter G: 5 clinics
Letter H: 7 clinics
Letter I: 7 clinics
Letter J: 6 clinics
Letter K: 3 clinics
Letter L: 28 clinics
Letter M: 4 clinics
Letter N: 5 clinics
Letter O: 4 clinics
Letter P: 5 clinics
Letter Q: 0 clinics
Letter R: 6 clinics
Letter S: 12 clinics
Letter T: 8 clinics
Letter U: 2 clinics
Letter V: 1 clinics
Letter W: 11 clinics
Letter X: 1 clinics
Letter Y: 1 clinics
Letter Z: 0 clinics

Total clinic listings: 214


,clinic_name,address,postcode,profile_url,website
0,Abbey Fertility Clinic,"6 Telford Court, Chestergates Business Park, C...",CH1 6LT,None,None
1,Aberdeen Fertility Centre,"Department of Obstetrics & Gynaecology, Aberde...",AB25 2ZL,https://www.hfea.gov.uk/choose-a-fertility-cli...,http://www.aberdeenfertility.org.uk/
2,Acorn Fertility,"112a, Cavendish Road, Colliers Wood, Merton, S...",SW19 2EZ,https://www.hfea.gov.uk/choose-a-fertility-cli...,None
3,Adrian Lower,"9, Upper Wimpole Street, London, W1G 6LJ",W1G 6LJ,None,None
4,Agora Clinic Brighton,"138, Dyke Road, Brighton, BN1 5PA",BN1 5PA,https://www.hfea.gov.uk/choose-a-fertility-cli...,http://www.agoraclinic.co.uk/


In [25]:
print("Total observations:", len(all_clinics_df))

print("\nMissing values:")
print(all_clinics_df.isna().sum())

print("\nDuplicate clinic names:",
      all_clinics_df["clinic_name"].duplicated().sum())

print("\nDuplicate clinic name and postcode combinations:",
      all_clinics_df.duplicated(
          subset=["clinic_name", "postcode"]
      ).sum())

display(
    all_clinics_df[
        all_clinics_df["postcode"].isna()
    ]
)

Total observations: 214

Missing values:
clinic_name      0
address          0
postcode         1
profile_url     95
website        126
dtype: int64

Duplicate clinic names: 3

Duplicate clinic name and postcode combinations: 3


,clinic_name,address,postcode,profile_url,website
197,TFP Wessex Fertility,"Unit 9, Berrywood Business Village, Tollbar Wa...",None,https://www.hfea.gov.uk/choose-a-fertility-cli...,http://www.wessexfertility.com/


In [26]:
print(
    all_clinics_df.loc[
        all_clinics_df["postcode"].isna(),
        ["clinic_name", "address"]
    ].to_string(index=False)
)

         clinic_name                                                                                           address
TFP Wessex Fertility Unit 9, Berrywood Business Village, Tollbar Way, Southampton, Hampshire, S030 2UN, United Kingdom


In [27]:
duplicates = all_clinics_df[
    all_clinics_df.duplicated(
        subset=["clinic_name", "postcode"],
        keep=False
    )
]

display(
    duplicates.sort_values(["clinic_name", "postcode"])
)

,clinic_name,address,postcode,profile_url,website
94,Fiona Fertility,"International House, 6 South Molton Street, Ma...",W1K 5QF,None,None
95,Fiona Fertility,"International House, 6 South Molton Street, Lo...",W1K 5QF,None,None
111,Ipswich IVF,"Ipswich Hospital, Heath Road, Ipswich, IP4 5PD",IP4 5PD,None,None
112,Ipswich IVF,"Ipswich Hospital, Heath Road, Ipswich, IP4 5PD",IP4 5PD,None,None
165,Oxford Reproductive Health (OxRH),"Level 4, Women's Centre, John Radcliffe Hospit...",OX3 9DU,None,None
166,Oxford Reproductive Health (OxRH),"Oxford Reproductive Health (OxRH), Headley Way...",OX3 9DU,None,None


In [28]:
# Correct the apparent character error in the postcode
all_clinics_df.loc[
    all_clinics_df["clinic_name"] == "TFP Wessex Fertility",
    "postcode"
] = "SO30 2UN"

print("Missing postcodes:", all_clinics_df["postcode"].isna().sum())

Missing postcodes: 0


In [29]:
# Preserve the original scraped data
clinics_clean_df = all_clinics_df.copy()

# Remove duplicate clinic name + postcode combinations
clinics_clean_df = clinics_clean_df.drop_duplicates(
    subset=["clinic_name", "postcode"],
    keep="first"
).reset_index(drop=True)

print("Original observations:", len(all_clinics_df))
print("Cleaned observations:", len(clinics_clean_df))

print("\nMissing values:")
print(clinics_clean_df.isna().sum())

Original observations: 214
Cleaned observations: 211

Missing values:
clinic_name      0
address          0
postcode         0
profile_url     92
website        123
dtype: int64


In [30]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)

all_clinics_df.to_csv("data/hfea_clinics_raw.csv", index=False)
clinics_clean_df.to_csv("data/hfea_clinics_clean.csv", index=False)

In [31]:
import requests
import pandas as pd
import time

# Standardize postcode formatting
clinics_clean_df["postcode"] = (
    clinics_clean_df["postcode"]
    .str.upper()
    .str.replace(" ", "", regex=False)
    .str.replace(r"^(.+)(.{3})$", r"\1 \2", regex=True)
)

display(clinics_clean_df[["clinic_name", "postcode"]].head(10))

,clinic_name,postcode
0,Abbey Fertility Clinic,CH1 6LT
1,Aberdeen Fertility Centre,AB25 2ZL
2,Acorn Fertility,SW19 2EZ
3,Adrian Lower,W1G 6LJ
4,Agora Clinic Brighton,BN1 5PA
5,Agora Clinic Eastbourne,BN26 6JF
6,Agora Clinic Worthing,BN12 4FD
7,Alma Fertility,SW11 8DD
8,Andrology Solutions,W1G 9RT
9,"Andrology Unit, Hammersmith Hospital",W12 0HS


In [32]:
postcodes = clinics_clean_df["postcode"].dropna().unique().tolist()

geo_data = []

# Process postcodes in batches of 100
for i in range(0, len(postcodes), 100):

    batch = postcodes[i:i+100]

    response = requests.post(
        "https://api.postcodes.io/postcodes",
        json={"postcodes": batch},
        timeout=30
    )

    response.raise_for_status()

    for item in response.json()["result"]:

        postcode = item["query"]
        result = item["result"]

        if result:
            geo_data.append({
                "postcode": postcode,
                "local_authority": result["admin_district"],
                "local_authority_code": result["codes"]["admin_district"],
                "country": result["country"],
                "latitude": result["latitude"],
                "longitude": result["longitude"]
            })

        else:
            geo_data.append({
                "postcode": postcode,
                "local_authority": None,
                "local_authority_code": None,
                "country": None,
                "latitude": None,
                "longitude": None
            })

    time.sleep(2)

geo_df = pd.DataFrame(geo_data)

display(geo_df.head())
print("Postcodes checked:", len(geo_df))

,postcode,local_authority,local_authority_code,country,latitude,longitude
0,CH1 6LT,Cheshire West and Chester,E06000050,England,53.246752,-2.932817
1,AB25 2ZL,Aberdeen City,S12000033,Scotland,57.155245,-2.132080
2,SW19 2EZ,Merton,E09000024,England,51.418616,-0.172276
3,W1G 6LJ,Westminster,E09000033,England,51.521066,-0.149078
4,BN1 5PA,Brighton and Hove,E06000043,England,50.834805,-0.150779


Postcodes checked: 200


In [33]:
clinics_geo_df = clinics_clean_df.merge(
    geo_df,
    on="postcode",
    how="left",
    validate="many_to_one"
)

print("Number of clinics:", len(clinics_geo_df))

print("\nMissing geographic information:")
print(
    clinics_geo_df[
        ["local_authority", "local_authority_code",
         "country", "latitude", "longitude"]
    ].isna().sum()
)

print("\nClinics by country:")
print(clinics_geo_df["country"].value_counts(dropna=False))

Number of clinics: 211

Missing geographic information:
local_authority         3
local_authority_code    3
country                 3
latitude                6
longitude               6
dtype: int64

Clinics by country:
country
England             181
Scotland             12
Wales                 9
Channel Islands       3
None                  3
Northern Ireland      3
Name: count, dtype: int64


In [34]:
unmatched = clinics_geo_df[
    clinics_geo_df["latitude"].isna()
]

display(
    unmatched[
        ["clinic_name", "address", "postcode"]
    ]
)

,clinic_name,address,postcode
13,"Assisted Reproduction Unit, Jersey General Hos...","Assisted Reproductive Unit, Enid Quenault Heal...",JE3 8JW
51,"Centre for Reproduction & Gynaecology Wales, S...","Penllergaer Business Park, Unit 4, Swansea, S4...",S4A 9HJ
117,Jersey Assisted Reproduction & Gynaecology Ser...,"Rozel Ward, 7th Floor, Rozel Consulting Rooms,...",JE1 3QS
118,Jersey Women’s Health Limited,"3 Highstead, La Rue Du Tapon, St. Saviour, Jer...",JR2 3RA
138,London Women's Clinic - Canterbury,"72, St. Dunstans Street, Canterbury, CT2 8BL",CT2 8BL
153,Medical Specialists Group,"Medical Specialists Group, Ruette Bray, Guerns...",GY1 3EX


In [35]:
for postcode in ["SA4 9HJ", "CT2 8BL"]:

    url = "https://api.postcodes.io/terminated_postcodes/" + postcode.replace(" ", "")

    response = requests.get(url, timeout=15)

    print("\nPostcode:", postcode)
    print("Status:", response.status_code)
    print("Result:", response.json().get("result"))


Postcode: SA4 9HJ
Status: 404
Result: None

Postcode: CT2 8BL
Status: 200
Result: {'postcode': 'CT2 8BL', 'year_terminated': 2017, 'month_terminated': 9, 'eastings': 614454, 'northings': 158170, 'longitude': 1.073795, 'latitude': 51.282289}


In [36]:
display(
    clinics_geo_df.loc[
        clinics_geo_df["postcode"] == "JR2 3RA",
        ["clinic_name", "address", "postcode"]
    ]
)

,clinic_name,address,postcode
118,Jersey Women’s Health Limited,"3 Highstead, La Rue Du Tapon, St. Saviour, Jer...",JR2 3RA


In [37]:
mask = clinics_geo_df["postcode"] == "CT2 8BL"

clinics_geo_df.loc[mask, "latitude"] = 51.282289
clinics_geo_df.loc[mask, "longitude"] = 1.073795

# Track where coordinates came from
clinics_geo_df["coordinate_source"] = "active_postcode"

clinics_geo_df.loc[
    clinics_geo_df["postcode"] == "CT2 8BL",
    "coordinate_source"
] = "terminated_postcode"

clinics_geo_df.loc[
    clinics_geo_df["latitude"].isna(),
    "coordinate_source"
] = None

print("Clinics without coordinates:",
      clinics_geo_df["latitude"].isna().sum())

Clinics without coordinates: 5


In [38]:
display(
    clinics_geo_df.loc[
        clinics_geo_df["postcode"] == "SA4 9HJ",
        ["clinic_name", "address", "postcode", "profile_url"]
    ]
)

,clinic_name,address,postcode,profile_url


In [39]:
print(
    clinics_geo_df.loc[
        clinics_geo_df["postcode"] == "JR2 3RA",
        "address"
    ].iloc[0]
)

3 Highstead, La Rue Du Tapon, St. Saviour, Jersey, JR2 3RA


In [40]:
display(
    clinics_geo_df.loc[
        clinics_geo_df["clinic_name"].str.contains(
            "Centre for Reproduction", case=False, na=False
        ),
        ["clinic_name", "address", "postcode", "profile_url"]
    ]
)

,clinic_name,address,postcode,profile_url
50,"Centre for Reproduction & Gynaecology Wales, B...","CRGW Bristol, 10 Apex Court, Woodlands, Bradle...",BS32 4JT,None
51,"Centre for Reproduction & Gynaecology Wales, S...","Penllergaer Business Park, Unit 4, Swansea, S4...",S4A 9HJ,None


In [41]:
clinics_geo_df["current_location_verified"] = (
    clinics_geo_df["coordinate_source"] == "active_postcode"
)

In [42]:
# Flag postcodes that need further verification
clinics_geo_df["postcode_issue"] = None

clinics_geo_df.loc[
    clinics_geo_df["postcode"] == "S4A 9HJ",
    "postcode_issue"
] = "Unmatched postcode - Swansea"

clinics_geo_df.loc[
    clinics_geo_df["postcode"] == "JR2 3RA",
    "postcode_issue"
] = "Unusual Jersey postcode"

clinics_geo_df.loc[
    clinics_geo_df["postcode"] == "CT2 8BL",
    "postcode_issue"
] = "Terminated postcode (September 2017)"

display(
    clinics_geo_df.loc[
        clinics_geo_df["postcode_issue"].notna(),
        ["clinic_name", "postcode", "postcode_issue"]
    ]
)

,clinic_name,postcode,postcode_issue
51,"Centre for Reproduction & Gynaecology Wales, S...",S4A 9HJ,Unmatched postcode - Swansea
118,Jersey Women’s Health Limited,JR2 3RA,Unusual Jersey postcode
138,London Women's Clinic - Canterbury,CT2 8BL,Terminated postcode (September 2017)


In [43]:
clinics_geo_df.to_csv(
    "data/hfea_clinics_geographic.csv",
    index=False
)

In [44]:
from selenium import webdriver

driver = webdriver.Chrome()

# Open Aberdeen Fertility Centre's HFEA profile
url = clinics_geo_df.loc[
    clinics_geo_df["clinic_name"] == "Aberdeen Fertility Centre",
    "profile_url"
].iloc[0]

driver.get(url)

print("Page title:", driver.title)

# Inspect the main headings
headings = driver.find_elements("xpath", "//h2 | //h3")

for heading in headings:
    print(heading.tag_name, "-", heading.text)

Page title: Clinic profile for Aberdeen Fertility Centre | HFEA
h3 - HFEA.gov.uk uses cookies to deliver content and make improvements to the website.
h2 - Inspection rating i
h2 - Patient rating i
h2 - IVF Birth Rate
h3 - 
h3 - Births per embryo transferred – excluding donor eggs and PGT-A
h3 - 
h3 - 
h2 - 
h2 - 
h2 - 
h2 - 
h2 - 
h2 - 
h2 - 
h2 - 
h2 - 
h2 - Get information
h2 - About the HFEA
h2 - For clinics
h2 - Research and data


In [45]:
# Get the visible text from the page
page_text = driver.find_element("xpath", "//body").text

# Search for treatment-related terms
keywords = [
    "Treatments",
    "Services",
    "IVF",
    "IUI",
    "Donor",
    "Egg freezing",
    "Fertility preservation",
    "Same-sex",
    "LGBT"
]

for keyword in keywords:
    print(f"{keyword}: {keyword.lower() in page_text.lower()}")

Treatments: True
Services: False
IVF: True
IUI: True
Donor: True
Egg freezing: False
Fertility preservation: True
Same-sex: False
LGBT: False


In [46]:
# Print text surrounding relevant terms
for keyword in ["Treatments", "Services", "IUI", "Donor"]:

    position = page_text.lower().find(keyword.lower())

    if position != -1:
        print(f"\n--- {keyword} ---")
        print(page_text[max(0, position-200):position+600])


--- Treatments ---
HFEA.gov.uk uses cookies to deliver content and make improvements to the website.
Accept cookies
Configure cookies
Home HFEA dashboard Contact us
Search for a treatment
Search
I am...
Treatments
Donation
Choose a clinic
About us
Read our latest report on The Fertility Sector 2025/26 here.
Compare +
Aberdeen Fertility Centre
Aberdeen Fertility Centre has been licensed by us since 1992 and offers all fertility treatments.
Inspection rating i
5 / 5
View inspection rating
Patient rating i
5 / 5 based on 14 ratings
Rate this clinic
View patient rating
IVF Birth Rate
View birth statistics
How do our inspectors rate the clinic?
We inspect all clinics against our standards. This clinic's inspection rating is:
5
5 / 5
All clinics must have a licence to operate. Our inspectors visit 

--- IUI ---
patients and the potentially less reliable the results.
Our embryo transfer data allow you to search for more age groups, and to look at pregnancy and birth rates in more detail.
Wha

In [48]:
from selenium import webdriver

driver = webdriver.Chrome()

url = "https://www.hfea.gov.uk/choose-a-fertility-clinic/search/results/9328-acorn-fertility/"

driver.get(url)

print(driver.title)

Clinic profile for Acorn Fertility | HFEA


In [49]:
driver.get(
    "https://www.hfea.gov.uk/choose-a-fertility-clinic/search/results/9328-acorn-fertility/"
)

treatment_headings = driver.find_elements(
    "xpath",
    "//h2[normalize-space()='Treatments']"
)

print("Treatment headings found:", len(treatment_headings))

for heading in treatment_headings:
    print(
        heading.find_element(
            "xpath", ".."
        ).get_attribute("outerHTML")[:4000]
    )

Treatment headings found: 1
<div class="detail col-lg-4 col-md-6">
                <h2>Treatments </h2>
                <ul class="list-unstyled">
                        <li>
                            
IVF
    <a href="#" data-toggle="tooltip" title="A type of fertility treatment where fertilisation takes place outside of the body. It’s suitable for people with a wide range of fertility issues and is one of the most commonly used and successful treatments available.&nbsp;See our Treatments section for more information.">
        <span class="hidden">Description</span>
        <span class="tooltip-icon" aria-hidden="true">i</span>
        <span class="sr-only">Information: A type of fertility treatment where fertilisation takes place outside of the body. It’s suitable for people with a wide range of fertility issues and is one of the most commonly used and successful treatments available.&nbsp;See our Treatments section for more information.</span>
    </a>

                        <

In [50]:
tables = driver.find_elements("xpath", "//table")

print("Number of tables:", len(tables))

for i, table in enumerate(tables):
    print(f"\n--- TABLE {i} ---")
    print(table.text[:700])

Number of tables: 7

--- TABLE 0 ---
5 10 15 20 25 30 35 40 45 50 55 60 65 70 75 80 85 90 95 100

--- TABLE 1 ---


--- TABLE 2 ---


--- TABLE 3 ---


--- TABLE 4 ---


--- TABLE 5 ---


--- TABLE 6 ---
Publication date: 26th January 2024
Last reviewed: 3rd October 2025


In [52]:
# Get the page's HTML
html = driver.page_source

# Search for the IVF birth rate section
keyword = "IVF Birth Rate"

position = html.lower().find(keyword.lower())

print("Found at position:", position)

if position != -1:
    print(html[max(0, position - 500):position + 3000])

Found at position: 41661
                    <span style="width: 50%" class="fill fill-part"></span>
                </div>
</div>
        </div>
        <p>
            4.5 / 5 based on 8 ratings
        </p>
            <a href="/choose-a-fertility-clinic/search/results/9328/patient-rating/">Rate this clinic</a>
    </div>
        <a class="scroll-to" href="#patient-ratings">View patient rating</a>
</div>

    <div class="col-lg-4">
        <div>

                    <h2>
                        <a href="#statistics">IVF Birth Rate</a>
                    </h2>

                <a class="search-page-link scroll-to btn btn-primary" style="background-color:#c8da2a;border:none;position:relative" href="#birth-statistics">View birth statistics</a>
        </div>

    </div>


        </div>
    </div>
</div>
    </div>
</div>

<div class="container">
    <h3 class="sr-only">Clinic profile accordion</h3>
    <div class="panel-group accordion clinic-accordion" id="accordion" role="tablist" 

In [53]:
# Find the birth statistics section
sections = driver.find_elements(
    "xpath",
    "//*[@id='birth-statistics']"
)

print("Sections found:", len(sections))

for section in sections:
    print("Tag:", section.tag_name)
    print("Text:")
    print(section.text[:3000])

Sections found: 1
Tag: div
Text:
What are the clinic's statistics?
We collect data about the treatments clinics carry out. See this clinic’s latest statistics below.
What is the clinic's IVF birth rate?
Find out a clinic’s IVF birth rates for:
Births per embryo transferred
Births per egg collection procedure
As well as the clinic’s multiple birth rate.
Whatever statistic you look at, it can’t tell you your individual chance of success (only your doctor can do that), but each statistic gives you an overall view of the clinic’s performance.
Births per embryo transferred
Births per embryo transferred is calculated using the number of births (twins or triplets are counted as one birth) divided by the number of embryos transferred, shown as a percentage, in a clinic over 12 months.
This statistic is not usually used to show outcomes over more than one IVF cycle (known as a cumulative success rate) because it counts embryo(s) transferred as a single event.
See the clinic’s rate below.
Find o

In [54]:
import re

section = driver.find_element(
    "xpath",
    "//*[@id='birth-statistics']"
)

# Find elements containing percentage signs
elements = section.find_elements(
    "xpath",
    ".//*[contains(text(), '%')]"
)

print("Elements containing percentages:", len(elements))

for element in elements[:30]:
    text = element.text.strip()

    if text:
        print(
            element.tag_name,
            "|",
            element.get_attribute("class"),
            "|",
            text[:150]
        )

Elements containing percentages: 5
p |  | For some people, having twins may seem like a wonderful thing, but multiple pregnancies are less safe for both patients and babies. That’s why we set 
span |  | 18%
span |  | 35%


In [55]:
section = driver.find_element(
    "xpath",
    "//*[@id='birth-statistics']"
)

spans = section.find_elements(
    "xpath",
    ".//span[contains(text(), '%')]"
)

for span in spans:
    if span.text.strip():
        print(span.text.strip())

18%
35%


In [56]:
clinic_rate = float(spans[0].text.strip().replace("%", ""))
national_rate = float(spans[1].text.strip().replace("%", ""))

rate_difference = clinic_rate - national_rate

print("Clinic rate:", clinic_rate)
print("National rate:", national_rate)
print("Difference:", rate_difference, "percentage points")

Clinic rate: 18.0
National rate: 35.0
Difference: -17.0 percentage points


In [57]:
section = driver.find_element(
    "xpath",
    "//*[@id='birth-statistics']"
)

spans = section.find_elements(
    "xpath",
    ".//span[contains(text(), '%')]"
)

for span in spans:
    if span.text.strip():
        print("\nPERCENTAGE:", span.text.strip())

        parent = span.find_element("xpath", "..")

        print("HTML:")
        print(parent.get_attribute("outerHTML")[:2000])


PERCENTAGE: 18%
HTML:
<div class="mean" style="left: 18%; display: inherit;">
                        <span>18%</span>
                    </div>

PERCENTAGE: 35%
HTML:
<div class="nat-ave" style="left: 35%;">
                        <span>35%</span>
                    </div>


In [58]:
from selenium.webdriver.common.by import By

section = driver.find_element(
    By.ID, "birth-statistics"
)

clinic_rate = section.find_element(
    By.CSS_SELECTOR, "div.mean span"
).text

national_rate = section.find_element(
    By.CSS_SELECTOR, "div.nat-ave span"
).text

print("Clinic birth rate:", clinic_rate)
print("National birth rate:", national_rate)

Clinic birth rate: 18%
National birth rate: 35%


In [59]:
clinic_rates = section.find_elements(
    By.CSS_SELECTOR, "div.mean span"
)

national_rates = section.find_elements(
    By.CSS_SELECTOR, "div.nat-ave span"
)

print("Clinic rates found:", len(clinic_rates))
print("National rates found:", len(national_rates))

for i, rate in enumerate(clinic_rates):
    print(f"Clinic rate {i + 1}:", rate.get_attribute("textContent").strip())

for i, rate in enumerate(national_rates):
    print(f"National rate {i + 1}:", rate.get_attribute("textContent").strip())

Clinic rates found: 3
National rates found: 3
Clinic rate 1: 18%
Clinic rate 2: 
Clinic rate 3: 
National rate 1: 35%
National rate 2: 13%
National rate 3: 16%


In [60]:
clinic_rates = section.find_elements(
    By.CSS_SELECTOR, "div.mean"
)

for i, rate in enumerate(clinic_rates):

    print(f"\n{'=' * 20}")
    print(f"STATISTIC {i + 1}")
    print(f"{'=' * 20}")

    print("Clinic rate:", rate.get_attribute("textContent").strip())

    parent = rate.find_element("xpath", "..")
    grandparent = parent.find_element("xpath", "..")

    print("\nSurrounding HTML:")
    print(grandparent.get_attribute("outerHTML")[:2500])


STATISTIC 1
Clinic rate: 18%

Surrounding HTML:
<div class="statistics-container">
                <div class="grid">
                    <table>
                        <tbody><tr class="label-row">
                                <td>5</td>
                                <td>10</td>
                                <td>15</td>
                                <td>20</td>
                                <td>25</td>
                                <td>30</td>
                                <td>35</td>
                                <td>40</td>
                                <td>45</td>
                                <td>50</td>
                                <td>55</td>
                                <td>60</td>
                                <td>65</td>
                                <td>70</td>
                                <td>75</td>
                                <td>80</td>
                                <td>85</td>
                                <td>90</td>
        

In [61]:
from selenium.webdriver.common.by import By

containers = section.find_elements(
    By.CSS_SELECTOR, ".statistics-container"
)

print("Statistics containers found:", len(containers))

for i, container in enumerate(containers):

    print(f"\n--- STATISTIC {i + 1} ---")

    # Get clinic and national rates
    clinic = container.find_elements(By.CSS_SELECTOR, ".mean span")
    national = container.find_elements(By.CSS_SELECTOR, ".nat-ave span")

    print(
        "Clinic:",
        clinic[0].get_attribute("textContent").strip()
        if clinic else "Not found"
    )

    print(
        "National:",
        national[0].get_attribute("textContent").strip()
        if national else "Not found"
    )

    # Look at the preceding HTML elements for labels
    previous = container.find_elements(
        By.XPATH, "./preceding-sibling::*[position() <= 3]"
    )

    for element in reversed(previous):
        print(
            "Previous element:",
            element.tag_name,
            "|",
            element.text[:300]
        )

Statistics containers found: 3

--- STATISTIC 1 ---
Clinic: 18%
National: 35%

--- STATISTIC 2 ---
Clinic: 
National: 13%

--- STATISTIC 3 ---
Clinic: 
National: 16%


In [62]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(driver.page_source, "html.parser")

charts = soup.select("#birth-statistics .statistics-container")

for i, chart in enumerate(charts, start=1):
    print(f"\n--- CHART {i} ---")

    # Find the nearest preceding heading
    heading = chart.find_previous(
        ["h2", "h3", "h4", "h5"]
    )

    print(
        "Nearest heading:",
        heading.get_text(" ", strip=True) if heading else "None"
    )

    # Find surrounding section text, excluding the chart
    parent = chart.parent

    for level in range(3):
        if parent is None:
            break

        text = parent.get_text(" ", strip=True)
        print(f"Parent level {level + 1}:", text[:350])

        parent = parent.parent


--- CHART 1 ---
Nearest heading: Births per embryo transferred – excluding donor eggs and PGT-A
Parent level 1: 5 10 15 20 25 30 35 40 45 50 55 60 65 70 75 80 85 90 95 100 18% 35% Clinic's rate National rate Reliability range Consistent with national average Description i Information: Taking into account the number of embryos in the calculation, this shows whether the clinic’s rate is below, consistent with or above the rate for all clinics across the UK. Sm
Parent level 2: 5 10 15 20 25 30 35 40 45 50 55 60 65 70 75 80 85 90 95 100 18% 35% Clinic's rate National rate Reliability range Consistent with national average Description i Information: Taking into account the number of embryos in the calculation, this shows whether the clinic’s rate is below, consistent with or above the rate for all clinics across the UK. Sm
Parent level 3: Births per embryo transferred – excluding donor eggs and PGT-A Under 38 38 and over All Births per egg collection – excluding donor eggs, including PGT-A

In [63]:
from selenium.webdriver.common.by import By

section = driver.find_element(
    By.ID, "birth-statistics"
)

radio_buttons = section.find_elements(
    By.CSS_SELECTOR, "input[type='radio']"
)

print("Radio buttons found:", len(radio_buttons))

for radio in radio_buttons[:12]:
    print(
        "Name:", radio.get_attribute("name"),
        "| Value:", radio.get_attribute("value"),
        "| ID:", radio.get_attribute("id"),
        "| Selected:", radio.is_selected()
    )

Radio buttons found: 15
Name: ivfembryo | Value: on | ID:  | Selected: True
Name: ivfembryo | Value: on | ID:  | Selected: False
Name: ivfembryo | Value: on | ID:  | Selected: False
Name: ivfegg | Value: on | ID:  | Selected: True
Name: ivfegg | Value: on | ID:  | Selected: False
Name: ivfegg | Value: on | ID:  | Selected: False
Name: ivfmult | Value: on | ID:  | Selected: True
Name: ivfmult | Value: on | ID:  | Selected: False
Name: ivfmult | Value: on | ID:  | Selected: False
Name: iuibirths | Value: on | ID:  | Selected: True
Name: iuibirths | Value: on | ID:  | Selected: False
Name: iuibirths | Value: on | ID:  | Selected: False


In [65]:
from selenium.webdriver.common.by import By
import pandas as pd
import numpy as np
import time

age_groups = ["Under 38", "38 and over", "All"]

outcome_groups = {
    "ivfembryo": ("IVF births per embryo transferred", 0),
    "iuibirths": ("IUI pregnancies per treatment", 1)
}

def extract_percentage(container, selector):
    elements = container.find_elements(By.CSS_SELECTOR, selector)

    if not elements:
        return np.nan

    value = elements[0].get_attribute("textContent").strip()
    value = value.replace("%", "")

    try:
        return float(value)
    except ValueError:
        return np.nan


results = []

for group, (outcome, chart_index) in outcome_groups.items():

    for i, age_group in enumerate(age_groups):

        buttons = driver.find_elements(
            By.CSS_SELECTOR,
            f"input[name='{group}']"
        )

        if len(buttons) != 3:
            print("Unexpected radio buttons:", group)
            continue

        # Select the age category
        driver.execute_script(
            "arguments[0].click();",
            buttons[i]
        )

        time.sleep(1)

        # Find the chart directly
        containers = driver.find_elements(
            By.CSS_SELECTOR,
            "#birth-statistics .statistics-container"
        )

        container = containers[chart_index]

        results.append({
            "clinic_name": "Acorn Fertility",
            "outcome": outcome,
            "age_group": age_group,
            "clinic_rate": extract_percentage(
                container, ".mean span"
            ),
            "national_rate": extract_percentage(
                container, ".nat-ave span"
            )
        })

age_stats_df = pd.DataFrame(results)

age_stats_df

,clinic_name,outcome,age_group,clinic_rate,national_rate
0,Acorn Fertility,IVF births per embryo transferred,Under 38,18.0,35.0
1,Acorn Fertility,IVF births per embryo transferred,38 and over,9.0,16.0
2,Acorn Fertility,IVF births per embryo transferred,All,14.0,29.0
3,Acorn Fertility,IUI pregnancies per treatment,Under 38,NaN,13.0
4,Acorn Fertility,IUI pregnancies per treatment,38 and over,NaN,6.0
5,Acorn Fertility,IUI pregnancies per treatment,All,NaN,11.0


In [66]:
section = driver.find_element(
    By.ID, "birth-statistics"
)

radio_buttons = section.find_elements(
    By.CSS_SELECTOR, "input[type='radio']"
)

group_names = list(dict.fromkeys(
    button.get_attribute("name")
    for button in radio_buttons
))

print("Outcome groups:", group_names)

Outcome groups: ['ivfembryo', 'ivfegg', 'ivfmult', 'iuibirths', 'dibirths']


In [68]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import pandas as pd
import numpy as np
import re
import time

outcome_groups = {
    "ivfembryo": "IVF births per embryo transferred",
    "ivfegg": "IVF births per egg collection",
    "ivfmult": "IVF multiple birth rate",
    "iuibirths": "IUI pregnancies per treatment",
    "dibirths": "Donor insemination births per treatment"
}

age_groups = ["Under 38", "38 and over", "All"]


def extract_percentage(container, selector):
    elements = container.find_elements(By.CSS_SELECTOR, selector)

    if not elements:
        return np.nan

    value = elements[0].get_attribute("textContent").strip()

    if not re.fullmatch(r"\d+(?:\.\d+)?%", value):
        return np.nan

    return float(value.replace("%", ""))


def scrape_clinic_statistics(driver, clinic_name, profile_url):

    driver.get(profile_url)

    WebDriverWait(driver, 20).until(
        lambda d: d.find_elements(By.ID, "birth-statistics")
    )

    results = []

    for group, outcome in outcome_groups.items():

        buttons = driver.find_elements(
            By.CSS_SELECTOR,
            f"#birth-statistics input[name='{group}']"
        )

        if len(buttons) != 3:
            print(f"Skipping {group} for {clinic_name}")
            continue

        for i, age_group in enumerate(age_groups):

            # Re-find elements to avoid stale references
            buttons = driver.find_elements(
                By.CSS_SELECTOR,
                f"#birth-statistics input[name='{group}']"
            )

            driver.execute_script(
                "arguments[0].click();",
                buttons[i]
            )

            time.sleep(0.5)

            # Match the selected radio group to its chart
            # using the chart ordering established on Acorn's page
            chart_groups = [
                "ivfembryo", "iuibirths", "dibirths"
            ]

            if group in chart_groups:
                chart_index = chart_groups.index(group)

                containers = driver.find_elements(
                    By.CSS_SELECTOR,
                    "#birth-statistics .statistics-container"
                )

                if len(containers) > chart_index:
                    container = containers[chart_index]

                    results.append({
                        "clinic_name": clinic_name,
                        "profile_url": profile_url,
                        "outcome": outcome,
                        "age_group": age_group,
                        "clinic_rate": extract_percentage(
                            container, ".mean span"
                        ),
                        "national_rate": extract_percentage(
                            container, ".nat-ave span"
                        )
                    })

    return results

In [69]:
test_results = scrape_clinic_statistics(
    driver,
    "Acorn Fertility",
    "https://www.hfea.gov.uk/choose-a-fertility-clinic/search/results/9328-acorn-fertility/"
)

test_df = pd.DataFrame(test_results)

test_df

,clinic_name,profile_url,outcome,age_group,clinic_rate,national_rate
0,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,Under 38,18.0,35.0
1,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,38 and over,9.0,16.0
2,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,All,14.0,29.0
3,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,Under 38,NaN,13.0
4,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,38 and over,NaN,6.0
5,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,All,NaN,11.0
6,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,Under 38,NaN,16.0
7,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,38 and over,NaN,7.0
8,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,All,NaN,14.0


In [70]:
# Find another clinic with a profile
test_clinic = clinics_geo_df[
    (clinics_geo_df["profile_url"].notna()) &
    (clinics_geo_df["clinic_name"] != "Acorn Fertility")
].iloc[0]

second_results = scrape_clinic_statistics(
    driver,
    test_clinic["clinic_name"],
    test_clinic["profile_url"]
)

second_test_df = pd.DataFrame(second_results)

second_test_df

,clinic_name,profile_url,outcome,age_group,clinic_rate,national_rate
0,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,Under 38,36.0,35.0
1,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,38 and over,18.0,16.0
2,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,All,32.0,29.0
3,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,Under 38,20.0,13.0
4,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,38 and over,20.0,6.0
5,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,All,17.0,11.0
6,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,Under 38,17.0,16.0
7,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,38 and over,17.0,7.0
8,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,All,17.0,14.0


In [71]:
from pathlib import Path
import pandas as pd
import time

Path("data").mkdir(exist_ok=True)

all_statistics = []
errors = []

# Only clinics with a profile URL
profiles = clinics_geo_df[
    clinics_geo_df["profile_url"].notna() &
    clinics_geo_df["profile_url"].str.strip().ne("")
].drop_duplicates(subset="profile_url")

print("Total clinics:", len(clinics_geo_df))
print("Profiles to scrape:", len(profiles))
print("Clinics without profiles:", len(clinics_geo_df) - len(profiles))

for i, row in enumerate(profiles.itertuples(index=False), start=1):

    try:
        clinic_results = scrape_clinic_statistics(
            driver,
            row.clinic_name,
            row.profile_url
        )

        all_statistics.extend(clinic_results)

        print(f"{i}/{len(profiles)}: {row.clinic_name} - {len(clinic_results)} rows")

    except Exception as e:
        errors.append({
            "clinic_name": row.clinic_name,
            "profile_url": row.profile_url,
            "error": str(e)
        })

        print(f"{i}/{len(profiles)}: ERROR - {row.clinic_name}")

    # Save after every clinic
    pd.DataFrame(all_statistics).to_csv(
        "data/hfea_clinic_statistics.csv",
        index=False
    )

    pd.DataFrame(errors).to_csv(
        "data/hfea_scraping_errors.csv",
        index=False
    )

    # Be respectful of the website
    time.sleep(5)

clinic_stats_df = pd.DataFrame(all_statistics)

print("\nFinished!")
print("Statistics rows:", len(clinic_stats_df))
print("Clinics with errors:", len(errors))

Total clinics: 211
Profiles to scrape: 119
Clinics without profiles: 92
1/119: Aberdeen Fertility Centre - 9 rows
2/119: Acorn Fertility - 9 rows
3/119: Agora Clinic Brighton - 9 rows
4/119: ERROR - Agora Clinic Eastbourne
5/119: Andrology Solutions - 9 rows
6/119: ERROR - Andrology Unit, Hammersmith Hospital
7/119: Aria Fertility - 9 rows
8/119: Assisted Reproduction and Gynaecology Centre - 9 rows
9/119: ERROR - Assisted Reproduction Unit (ARU), University Hospital of Hartlepool
10/119: Avenues - 9 rows
11/119: ERROR - Ayrshire Fertility Unit, University Hospital Crosshouse
12/119: Barts Health Centre for Reproductive Medicine - 9 rows
13/119: Beginnings at Epsom & St Helier NHS University Trust - 9 rows
14/119: Birmingham Women's Hospital - 9 rows
15/119: Bourn Hall Clinic - 9 rows
16/119: Bourn Hall Clinic Norwich - 9 rows
17/119: Bourn Hall Clinic Wickford - 9 rows
18/119: Bristol Centre for Reproductive Medicine - 9 rows
19/119: Cambridge IVF - 9 rows
20/119: Care Fertility Bath 

In [72]:
errors_df = pd.read_csv("data/hfea_scraping_errors.csv")

for _, row in errors_df.iterrows():
    print(f"\n{row['clinic_name']}")
    print(str(row["error"])[:250])
    print("-" * 50)


Agora Clinic Eastbourne
Message: 

--------------------------------------------------

Andrology Unit, Hammersmith Hospital
Message: 

--------------------------------------------------

Assisted Reproduction Unit (ARU), University Hospital of Hartlepool
Message: 

--------------------------------------------------

Ayrshire Fertility Unit, University Hospital Crosshouse
Message: 

--------------------------------------------------

Fertility Fusion
Message: 

--------------------------------------------------

Future Health Biobank
Message: 

--------------------------------------------------

Genus Medical Fertility Edinburgh
Message: 

--------------------------------------------------

Gloucestershire Hospitals NHS Trust
Message: 

--------------------------------------------------

Jack Copland Centre, Scottish National Blood Transfusion Service (SNBTS)
Message: 

--------------------------------------------------

King's Fertility-Harley Street
Message: 

-----------------------

In [73]:
clinic_stats_df = pd.read_csv(
    "data/hfea_clinic_statistics.csv"
)

summary = (
    clinic_stats_df
    .groupby(["outcome", "age_group"])
    .agg(
        total_clinics=("clinic_name", "nunique"),
        available_rates=("clinic_rate", "count"),
        mean_clinic_rate=("clinic_rate", "mean"),
        median_clinic_rate=("clinic_rate", "median")
    )
    .reset_index()
)

summary

,outcome,age_group,total_clinics,available_rates,mean_clinic_rate,median_clinic_rate
0,Donor insemination births per treatment,38 and over,98,77,9.454545,7.0
1,Donor insemination births per treatment,All,98,79,14.708861,15.0
2,Donor insemination births per treatment,Under 38,98,74,17.635135,17.0
3,IUI pregnancies per treatment,38 and over,98,53,9.433962,7.0
4,IUI pregnancies per treatment,All,98,58,12.206897,10.5
5,IUI pregnancies per treatment,Under 38,98,50,14.940000,13.0
6,IVF births per embryo transferred,38 and over,98,90,16.177778,16.0
7,IVF births per embryo transferred,All,98,91,28.296703,29.0
8,IVF births per embryo transferred,Under 38,98,90,33.855556,35.0


In [77]:
failed_clinic = errors_df.iloc[0]

print("Clinic:", failed_clinic["clinic_name"])

driver.get(failed_clinic["profile_url"])

print("Page title:", driver.title)

print(
    "Statistics sections:",
    len(driver.find_elements(By.ID, "birth-statistics"))
)

print(
    "Page headings:",
    [h.text for h in driver.find_elements(By.TAG_NAME, "h1")]
)

Clinic: Agora Clinic Eastbourne


Page title: Clinic profile for Agora Clinic Eastbourne | HFEA
Statistics sections: 0
Page headings: ['Agora Clinic Eastbourne']


In [75]:
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()

In [78]:
import time
import pandas as pd
from selenium.webdriver.common.by import By

diagnostics = []

for i, row in errors_df.iterrows():

    try:
        driver.get(row["profile_url"])

        time.sleep(2)

        has_statistics = len(
            driver.find_elements(By.ID, "birth-statistics")
        ) > 0

        diagnostics.append({
            "clinic_name": row["clinic_name"],
            "has_statistics_section": has_statistics,
            "page_title": driver.title
        })

        print(
            row["clinic_name"],
            "— Statistics section:",
            has_statistics
        )

    except Exception as e:
        diagnostics.append({
            "clinic_name": row["clinic_name"],
            "has_statistics_section": None,
            "page_title": None
        })

        print(row["clinic_name"], "— Could not inspect")

    time.sleep(5)

diagnostics_df = pd.DataFrame(diagnostics)
diagnostics_df.to_csv(
    "data/hfea_failed_profiles_diagnostics.csv",
    index=False
)

diagnostics_df["has_statistics_section"].value_counts(
    dropna=False
)

Agora Clinic Eastbourne — Statistics section: False
Andrology Unit, Hammersmith Hospital — Statistics section: False
Assisted Reproduction Unit (ARU), University Hospital of Hartlepool — Statistics section: False
Ayrshire Fertility Unit, University Hospital Crosshouse — Statistics section: False
Fertility Fusion — Statistics section: False
Future Health Biobank — Statistics section: False
Genus Medical Fertility Edinburgh — Statistics section: False
Gloucestershire Hospitals NHS Trust — Statistics section: False
Jack Copland Centre, Scottish National Blood Transfusion Service (SNBTS) — Statistics section: False
King's Fertility-Harley Street — Statistics section: False
Lanarkshire Acute Hospital NHS Trust — Statistics section: False
London Sperm Bank (LSB) London Bridge — Statistics section: False
Orian Gametes — Statistics section: False
Royal Derby Hospital — Statistics section: False
Royal Surrey Hospital NHS Foundation Trust — Statistics section: False
Semovo Glasgow — Statistics s

has_statistics_section
False    21
Name: count, dtype: int64

In [79]:
# Start with our complete geographic dataset
clinics_final_df = clinics_geo_df.copy()

# Identify profiles where statistics were collected
successful_urls = set(
    clinic_stats_df["profile_url"].dropna()
)

# Identify profiles without the statistics section
no_stats_urls = set(
    diagnostics_df.loc[
        diagnostics_df["has_statistics_section"] == False,
        "clinic_name"
    ]
)

def classify_statistics(row):
    if pd.isna(row["profile_url"]):
        return "No profile URL"
    elif row["profile_url"] in successful_urls:
        return "Statistics section available"
    elif row["clinic_name"] in no_stats_urls:
        return "No statistics section"
    else:
        return "Not verified"

clinics_final_df["statistics_status"] = (
    clinics_final_df.apply(classify_statistics, axis=1)
)

print(clinics_final_df["statistics_status"].value_counts())

clinics_final_df.to_csv(
    "data/hfea_clinics_final.csv",
    index=False
)

statistics_status
Statistics section available    98
No profile URL                  92
No statistics section           21
Name: count, dtype: int64


In [80]:
driver.get(
    "https://www.hfea.gov.uk/choose-a-fertility-clinic/search/results/9328-acorn-fertility/"
)

from selenium.webdriver.common.by import By

section = driver.find_element(By.ID, "birth-statistics")

print(section.text[:5000])

What are the clinic's statistics?
We collect data about the treatments clinics carry out. See this clinic’s latest statistics below.
What is the clinic's IVF birth rate?
Find out a clinic’s IVF birth rates for:
Births per embryo transferred
Births per egg collection procedure
As well as the clinic’s multiple birth rate.
Whatever statistic you look at, it can’t tell you your individual chance of success (only your doctor can do that), but each statistic gives you an overall view of the clinic’s performance.
Births per embryo transferred
Births per embryo transferred is calculated using the number of births (twins or triplets are counted as one birth) divided by the number of embryos transferred, shown as a percentage, in a clinic over 12 months.
This statistic is not usually used to show outcomes over more than one IVF cycle (known as a cumulative success rate) because it counts embryo(s) transferred as a single event.
See the clinic’s rate below.
Find out more about this statistic here

In [81]:
from selenium.webdriver.common.by import By

section = driver.find_element(By.ID, "birth-statistics")

date_elements = section.find_elements(
    By.XPATH,
    ".//*[contains(text(), '01/01/2024')]"
)

print("Matching elements:", len(date_elements))

for element in date_elements[:5]:
    print("\nTAG:", element.tag_name)
    print("CLASS:", element.get_attribute("class"))
    print("HTML:", element.get_attribute("outerHTML")[:1000])

Matching elements: 2

TAG: span
CLASS: period dateperiod
HTML: <span class="period dateperiod">01/01/2024 to 31/12/2024</span>

TAG: span
CLASS: period
HTML: <span class="period">01/01/2024 to 31/12/2024</span>


In [82]:
section = driver.find_element(By.ID, "birth-statistics")

period_elements = section.find_elements(
    By.CSS_SELECTOR,
    "span.period"
)

for i, element in enumerate(period_elements):
    print(
        i,
        "| Class:", element.get_attribute("class"),
        "| Period:", element.get_attribute("textContent").strip()
    )

0 | Class: period dateperiod | Period: 01/01/2024 to 31/12/2024
1 | Class: period | Period: 01/01/2025 to 31/12/2025
2 | Class: period | Period: 01/01/2024 to 31/12/2024


In [83]:
import re
import pandas as pd
from selenium.webdriver.common.by import By

def extract_reporting_periods(driver, clinic_name, profile_url):

    section = driver.find_element(By.ID, "birth-statistics")

    periods = section.find_elements(
        By.CSS_SELECTOR,
        "span.period"
    )

    outcomes = [
        "IVF births per embryo transferred",
        "IUI pregnancies per treatment",
        "Donor insemination births per treatment"
    ]

    results = []

    if len(periods) != 3:
        print(f"Unexpected number of periods for {clinic_name}: {len(periods)}")
        return results

    for outcome, element in zip(outcomes, periods):

        period_text = element.get_attribute("textContent").strip()

        dates = re.findall(
            r"\d{2}/\d{2}/\d{4}",
            period_text
        )

        if len(dates) == 2:
            results.append({
                "clinic_name": clinic_name,
                "profile_url": profile_url,
                "outcome": outcome,
                "period_start": pd.to_datetime(
                    dates[0], format="%d/%m/%Y"
                ),
                "period_end": pd.to_datetime(
                    dates[1], format="%d/%m/%Y"
                )
            })

    return results

In [84]:
test_periods = extract_reporting_periods(
    driver,
    "Acorn Fertility",
    driver.current_url
)

pd.DataFrame(test_periods)

,clinic_name,profile_url,outcome,period_start,period_end
0,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,2024-01-01,2024-12-31
1,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,2025-01-01,2025-12-31
2,Acorn Fertility,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,2024-01-01,2024-12-31


In [85]:
# Select Aberdeen Fertility Centre
second_clinic = clinic_stats_df[
    clinic_stats_df["clinic_name"] == "Aberdeen Fertility Centre"
].iloc[0]

driver.get(second_clinic["profile_url"])

# Extract reporting periods
second_periods = extract_reporting_periods(
    driver,
    second_clinic["clinic_name"],
    second_clinic["profile_url"]
)

pd.DataFrame(second_periods)

WebDriverException: Message: unknown error: net::ERR_NAME_NOT_RESOLVED
  (Session info: chrome=155.0.8059.85)
Stacktrace:
0   chromedriver                        0x0000000100998884 chromedriver + 3541124
1   chromedriver                        0x000000010067d0d0 chromedriver + 282832
2   chromedriver                        0x000000010067a708 chromedriver + 272136
3   chromedriver                        0x000000010066d78c chromedriver + 219020
4   chromedriver                        0x000000010066f11c chromedriver + 225564
5   chromedriver                        0x000000010066dc04 chromedriver + 220164
6   chromedriver                        0x000000010066d3b0 chromedriver + 218032
7   chromedriver                        0x000000010066d0e0 chromedriver + 217312
8   chromedriver                        0x000000010066aec4 chromedriver + 208580
9   chromedriver                        0x000000010066b96c chromedriver + 211308
10  chromedriver                        0x00000001006805ac chromedr

In [86]:
print(repr(second_clinic["profile_url"]))

'https://www.hfea.gov.uk/choose-a-fertility-clinic/search/results/19-aberdeen-fertility-centre/'


In [87]:
print("Current URL:", driver.current_url)
print("Page title:", driver.title)

Current URL: https://www.hfea.gov.uk/choose-a-fertility-clinic/search/results/19-aberdeen-fertility-centre/
Page title: www.hfea.gov.uk


In [88]:
driver.get(second_clinic["profile_url"])

second_periods = extract_reporting_periods(
    driver,
    second_clinic["clinic_name"],
    second_clinic["profile_url"]
)

pd.DataFrame(second_periods)

,clinic_name,profile_url,outcome,period_start,period_end
0,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,2024-01-01,2024-12-31
1,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,2025-01-01,2025-12-31
2,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,2024-01-01,2024-12-31


In [89]:
driver.get(second_clinic["profile_url"])

second_periods = extract_reporting_periods(
    driver,
    second_clinic["clinic_name"],
    second_clinic["profile_url"]
)

pd.DataFrame(second_periods)

,clinic_name,profile_url,outcome,period_start,period_end
0,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,2024-01-01,2024-12-31
1,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,2025-01-01,2025-12-31
2,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,2024-01-01,2024-12-31


In [90]:
import time
import pandas as pd
from pathlib import Path

Path("data").mkdir(exist_ok=True)

all_periods = []
period_errors = []

# Only profiles for which we collected statistics
successful_profiles = (
    clinic_stats_df[["clinic_name", "profile_url"]]
    .drop_duplicates(subset="profile_url")
    .reset_index(drop=True)
)

print("Profiles to process:", len(successful_profiles))

for i, row in enumerate(
    successful_profiles.itertuples(index=False), start=1
):
    try:
        driver.get(row.profile_url)

        clinic_periods = extract_reporting_periods(
            driver,
            row.clinic_name,
            row.profile_url
        )

        if len(clinic_periods) != 3:
            raise ValueError(
                f"Expected 3 reporting periods, got {len(clinic_periods)}"
            )

        all_periods.extend(clinic_periods)

        print(f"{i}/{len(successful_profiles)}: "
              f"{row.clinic_name} - 3 periods")

    except Exception as e:
        period_errors.append({
            "clinic_name": row.clinic_name,
            "profile_url": row.profile_url,
            "error": str(e)
        })

        print(f"{i}/{len(successful_profiles)}: "
              f"ERROR - {row.clinic_name}")

    # Save progress after every clinic
    pd.DataFrame(all_periods).to_csv(
        "data/hfea_reporting_periods.csv",
        index=False
    )

    pd.DataFrame(period_errors).to_csv(
        "data/hfea_reporting_period_errors.csv",
        index=False
    )

    # Pause between requests
    time.sleep(8)

periods_df = pd.DataFrame(all_periods)

print("\nFinished!")
print("Reporting-period rows:", len(periods_df))
print("Profiles with errors:", len(period_errors))

Profiles to process: 98
1/98: Aberdeen Fertility Centre - 3 periods
2/98: Acorn Fertility - 3 periods
3/98: Agora Clinic Brighton - 3 periods
4/98: Andrology Solutions - 3 periods
5/98: Aria Fertility - 3 periods
6/98: Assisted Reproduction and Gynaecology Centre - 3 periods
7/98: Avenues - 3 periods
8/98: Barts Health Centre for Reproductive Medicine - 3 periods
9/98: Beginnings at Epsom & St Helier NHS University Trust - 3 periods
10/98: Birmingham Women's Hospital - 3 periods
11/98: Bourn Hall Clinic - 3 periods
12/98: Bourn Hall Clinic Norwich - 3 periods
13/98: Bourn Hall Clinic Wickford - 3 periods
14/98: Bristol Centre for Reproductive Medicine - 3 periods
15/98: Cambridge IVF - 3 periods
16/98: Care Fertility Bath - 3 periods
17/98: Care Fertility Birmingham - 3 periods
18/98: Care Fertility Cardiff - 3 periods
19/98: Care Fertility Cheshire - 3 periods
20/98: Care Fertility Leeds - 3 periods
21/98: Care Fertility Liverpool - 3 periods
22/98: Care Fertility London - 3 periods
2

In [91]:
# Reload our saved datasets
clinic_stats_df = pd.read_csv(
    "data/hfea_clinic_statistics.csv"
)

periods_df = pd.read_csv(
    "data/hfea_reporting_periods.csv"
)

# Merge reporting periods into the statistics dataset
clinic_stats_final_df = clinic_stats_df.merge(
    periods_df[
        ["profile_url", "outcome", "period_start", "period_end"]
    ],
    on=["profile_url", "outcome"],
    how="left",
    validate="many_to_one"
)

print("Original observations:", len(clinic_stats_df))
print("Merged observations:", len(clinic_stats_final_df))

print("\nMissing reporting periods:")
print(
    clinic_stats_final_df[
        ["period_start", "period_end"]
    ].isna().sum()
)

clinic_stats_final_df.head(9)

Original observations: 882
Merged observations: 882

Missing reporting periods:
period_start    0
period_end      0
dtype: int64


,clinic_name,profile_url,outcome,age_group,clinic_rate,national_rate,period_start,period_end
0,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,Under 38,36.0,35.0,2024-01-01,2024-12-31
1,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,38 and over,18.0,16.0,2024-01-01,2024-12-31
2,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IVF births per embryo transferred,All,32.0,29.0,2024-01-01,2024-12-31
3,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,Under 38,20.0,13.0,2025-01-01,2025-12-31
4,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,38 and over,20.0,6.0,2025-01-01,2025-12-31
5,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,IUI pregnancies per treatment,All,17.0,11.0,2025-01-01,2025-12-31
6,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,Under 38,17.0,16.0,2024-01-01,2024-12-31
7,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,38 and over,17.0,7.0,2024-01-01,2024-12-31
8,Aberdeen Fertility Centre,https://www.hfea.gov.uk/choose-a-fertility-cli...,Donor insemination births per treatment,All,17.0,14.0,2024-01-01,2024-12-31


In [92]:
clinic_stats_final_df["reporting_year"] = (
    pd.to_datetime(
        clinic_stats_final_df["period_end"]
    ).dt.year
)

year_summary = (
    clinic_stats_final_df
    .groupby(["outcome", "reporting_year"])["profile_url"]
    .nunique()
    .reset_index(name="number_of_clinics")
)

year_summary

,outcome,reporting_year,number_of_clinics
0,Donor insemination births per treatment,2023,2
1,Donor insemination births per treatment,2024,96
2,IUI pregnancies per treatment,2024,2
3,IUI pregnancies per treatment,2025,96
4,IVF births per embryo transferred,2023,2
5,IVF births per embryo transferred,2024,96


In [93]:
older_periods = (
    clinic_stats_final_df[
        (
            (clinic_stats_final_df["outcome"] == "IVF births per embryo transferred") &
            (clinic_stats_final_df["reporting_year"] == 2023)
        )
    ][["clinic_name", "period_start", "period_end"]]
    .drop_duplicates()
)

older_periods

,clinic_name,period_start,period_end
747,Sussex Sperm Bank,2023-01-01,2023-12-31
855,"Wales Fertility Institute, Cardiff",2023-01-01,2023-12-31


In [94]:
older_clinics = [
    "Sussex Sperm Bank",
    "Wales Fertility Institute, Cardiff"
]

clinic_stats_final_df.loc[
    clinic_stats_final_df["clinic_name"].isin(older_clinics),
    [
        "clinic_name",
        "outcome",
        "age_group",
        "clinic_rate",
        "national_rate",
        "reporting_year"
    ]
].sort_values(
    ["clinic_name", "outcome", "age_group"]
)

,clinic_name,outcome,age_group,clinic_rate,national_rate,reporting_year
754,Sussex Sperm Bank,Donor insemination births per treatment,38 and over,NaN,7.0,2023
755,Sussex Sperm Bank,Donor insemination births per treatment,All,NaN,13.0,2023
753,Sussex Sperm Bank,Donor insemination births per treatment,Under 38,NaN,15.0,2023
751,Sussex Sperm Bank,IUI pregnancies per treatment,38 and over,NaN,NaN,2024
752,Sussex Sperm Bank,IUI pregnancies per treatment,All,NaN,NaN,2024
750,Sussex Sperm Bank,IUI pregnancies per treatment,Under 38,NaN,NaN,2024
748,Sussex Sperm Bank,IVF births per embryo transferred,38 and over,NaN,16.0,2023
749,Sussex Sperm Bank,IVF births per embryo transferred,All,NaN,28.0,2023
747,Sussex Sperm Bank,IVF births per embryo transferred,Under 38,NaN,34.0,2023
862,"Wales Fertility Institute, Cardiff",Donor insemination births per treatment,38 and over,NaN,7.0,2023


In [95]:
# Start with all 211 geographic clinic records
clinics_geo_final_df = clinics_final_df.copy()

# Merge statistics using profile URL
clinics_combined_df = clinics_geo_final_df.merge(
    clinic_stats_final_df[
        [
            "profile_url",
            "outcome",
            "age_group",
            "clinic_rate",
            "national_rate",
            "period_start",
            "period_end",
            "reporting_year"
        ]
    ],
    on="profile_url",
    how="left",
    validate="one_to_many"
)

print("Unique clinics:",
      clinics_combined_df[
          ["clinic_name", "postcode"]
      ].drop_duplicates().shape[0])

print("Total rows:", len(clinics_combined_df))

print("Clinics with at least one reported rate:",
      clinics_combined_df.loc[
          clinics_combined_df["clinic_rate"].notna(),
          "profile_url"
      ].nunique())

clinics_combined_df.head()

MergeError: Merge keys are not unique in left dataset; not a one-to-many merge

In [96]:
print("Total clinics:", len(clinics_final_df))

print(
    "Missing profile URLs:",
    clinics_final_df["profile_url"].isna().sum()
)

print(
    "Duplicate nonmissing profile URLs:",
    clinics_final_df.loc[
        clinics_final_df["profile_url"].notna(),
        "profile_url"
    ].duplicated().sum()
)

Total clinics: 211
Missing profile URLs: 92
Duplicate nonmissing profile URLs: 0


In [97]:
# Clinics with a profile URL
with_profiles = clinics_final_df[
    clinics_final_df["profile_url"].notna()
].copy()

# Clinics without a profile URL
without_profiles = clinics_final_df[
    clinics_final_df["profile_url"].isna()
].copy()

# Merge only clinics with profiles
merged_profiles = with_profiles.merge(
    clinic_stats_final_df[
        [
            "profile_url",
            "outcome",
            "age_group",
            "clinic_rate",
            "national_rate",
            "period_start",
            "period_end",
            "reporting_year"
        ]
    ],
    on="profile_url",
    how="left",
    validate="one_to_many"
)

# Add clinics without profiles back
clinics_combined_df = pd.concat(
    [merged_profiles, without_profiles],
    ignore_index=True
)

print("Total rows:", len(clinics_combined_df))

print(
    "Unique clinics:",
    clinics_combined_df[
        ["clinic_name", "postcode"]
    ].drop_duplicates().shape[0]
)

Total rows: 995
Unique clinics: 211


In [98]:
# Check duplicate clinic-outcome-age combinations
duplicates = clinics_combined_df[
    clinics_combined_df["outcome"].notna()
].duplicated(
    subset=["profile_url", "outcome", "age_group"]
).sum()

print("Duplicate clinic-outcome-age rows:", duplicates)

# Check geographic coverage
print(
    "Clinics with coordinates:",
    clinics_combined_df.loc[
        clinics_combined_df["latitude"].notna() &
        clinics_combined_df["longitude"].notna(),
        ["clinic_name", "postcode"]
    ].drop_duplicates().shape[0]
)

# Check outcome coverage
print(
    "Clinics with at least one success rate:",
    clinics_combined_df.loc[
        clinics_combined_df["clinic_rate"].notna(),
        "profile_url"
    ].nunique()
)

# Check whether rates fall within 0–100%
invalid_rates = clinics_combined_df[
    clinics_combined_df["clinic_rate"].notna() &
    ~clinics_combined_df["clinic_rate"].between(0, 100)
]

print("Invalid success rates:", len(invalid_rates))

# Save final combined dataset
clinics_combined_df.to_csv(
    "data/hfea_clinics_combined.csv",
    index=False
)

Duplicate clinic-outcome-age rows: 0
Clinics with coordinates: 206
Clinics with at least one success rate: 96
Invalid success rates: 0


In [99]:
# Create a clinic-level dataset for mapping
clinics_map_df = (
    clinics_combined_df
    .drop_duplicates(subset=["clinic_name", "postcode"])
    .copy()
)

# Keep clinics with valid coordinates
clinics_map_df = clinics_map_df.dropna(
    subset=["latitude", "longitude"]
)

print("Clinics ready to map:", len(clinics_map_df))

print("\nClinics by country:")
print(clinics_map_df["country"].value_counts())

print("\nPreview:")
display(
    clinics_map_df[
        [
            "clinic_name",
            "postcode",
            "country",
            "local_authority",
            "latitude",
            "longitude",
            "statistics_status"
        ]
    ].head()
)

Clinics ready to map: 206

Clinics by country:
country
England             181
Scotland             12
Wales                 9
Northern Ireland      3
Name: count, dtype: int64

Preview:


,clinic_name,postcode,country,local_authority,latitude,longitude,statistics_status
0,Aberdeen Fertility Centre,AB25 2ZL,Scotland,Aberdeen City,57.155245,-2.132080,Statistics section available
9,Acorn Fertility,SW19 2EZ,England,Merton,51.418616,-0.172276,Statistics section available
18,Agora Clinic Brighton,BN1 5PA,England,Brighton and Hove,50.834805,-0.150779,Statistics section available
27,Agora Clinic Eastbourne,BN26 6JF,England,Wealden,50.819678,0.268208,No statistics section
28,Andrology Solutions,W1G 9RT,England,Westminster,51.517464,-0.148200,Statistics section available


In [100]:
clinics_map_df.loc[
    clinics_map_df["country"].isna(),
    ["clinic_name", "postcode", "latitude", "longitude"]
]

,clinic_name,postcode,latitude,longitude
961,London Women's Clinic - Canterbury,CT2 8BL,51.282289,1.073795


In [101]:
# Identify the Canterbury clinic
canterbury_mask = (
    clinics_combined_df["clinic_name"] ==
    "London Women's Clinic - Canterbury"
)

# Fill the country without changing the coordinates
clinics_combined_df.loc[
    canterbury_mask, "country"
] = "England"

# Flag the historical postcode
clinics_combined_df.loc[
    canterbury_mask, "postcode_issue"
] = "Terminated postcode - historical coordinates"

# Save the updated dataset
clinics_combined_df.to_csv(
    "data/hfea_clinics_combined.csv",
    index=False
)

print(
    clinics_combined_df.loc[
        canterbury_mask,
        ["clinic_name", "postcode", "country", "postcode_issue"]
    ].drop_duplicates()
)

                            clinic_name postcode  country  \
961  London Women's Clinic - Canterbury  CT2 8BL  England   

                                   postcode_issue  
961  Terminated postcode - historical coordinates  


In [103]:
clinics_map_df = (
    clinics_combined_df
    .drop_duplicates(subset=["clinic_name", "postcode"])
    .dropna(subset=["latitude", "longitude"])
    .copy()
)

print(clinics_map_df["country"].value_counts())

country
England             182
Scotland             12
Wales                 9
Northern Ireland      3
Name: count, dtype: int64


In [105]:
clinic_level_df = (
    clinics_combined_df
    .drop_duplicates(subset=["clinic_name", "postcode"])
    .copy()
)

country_summary = (
    clinic_level_df["country"]
    .value_counts(dropna=False)
    .rename_axis("Country")
    .reset_index(name="Number of clinics")
)

country_summary["Percentage"] = (
    country_summary["Number of clinics"]
    / len(clinic_level_df) * 100
).round(2)

country_summary

,Country,Number of clinics,Percentage
0,England,182,86.26
1,Scotland,12,5.69
2,Wales,9,4.27
3,Northern Ireland,3,1.42
4,Channel Islands,3,1.42
5,None,2,0.95


In [106]:
local_authority_summary = (
    clinic_level_df
    .groupby(["country", "local_authority"], dropna=False)
    .size()
    .reset_index(name="number_of_clinics")
    .sort_values("number_of_clinics", ascending=False)
)

local_authority_summary.head(20)

,country,local_authority,number_of_clinics
85,England,Westminster,27
53,England,Liverpool,6
105,Wales,Cardiff,5
70,England,Southwark,5
62,England,Sheffield,5
3,England,Birmingham,4
58,England,Nottingham,4
51,England,Leeds,4
0,Channel Islands,Channel Islands,3
30,England,Eastleigh,3
